## JupyterLite compatibility notes

This version has two changes from the original so it runs cleanly on a static JupyterLite/GitHub Pages deployment:

1. **Removed `plt.rcParams["text.usetex"] = True`.** JupyterLite has no LaTeX toolchain in the browser, so this would error on every plot. All labels already use matplotlib's built-in mathtext (`$...$`), which renders fine without real LaTeX.
2. **Replaced the `fig.canvas` / ipympl-style live widget with a `widgets.Output()` redraw pattern.** The original relied on the interactive matplotlib widget backend, which currently has an open bug in JupyterLite/Pyodide (`ImportError: cannot import name 'alert' from 'js'`). The version below just redraws the static figure into an `Output` widget on every slider change — a bit less elegant, but it only needs the default inline backend and works reliably in-browser.

If deploying via `jupyter lite build`, make sure `requirements.txt` pins matching versions of `ipywidgets` and `jupyterlab_widgets` (e.g. `ipywidgets==8.1.2`, `jupyterlab_widgets==3.0.10`) so the sliders render.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import ipywidgets as widgets
from IPython.display import display


In [ ]:
def mono_w(k, kappa = 1, m = 1, a = 1):
    """
    Returns the frequency of normal modes of the 
    harmonic 1D monoatomic chain as a function of their
    wavevector. 
    
    k :    Normal mode wavevector
    kappa: Spring constant defining force between neighboring atoms
    m:     Atomic mass
    a:     Lattice spacing
    """
    
    return 2*np.sqrt(kappa/m)*np.abs(np.sin(k*a/2))

In [ ]:
def displacement(n, k, t, A=1, a=1):
    """
    Returns the displacement from equilibrium of the n-th 
    atom at time t for a system in the k-th phonon mode.
    
    """
    
    omega = mono_w(k)
    dx_n = a*np.exp(1j*omega*t-1j*k*n*a)
    return np.real(dx_n)

In [ ]:
kgrid = np.linspace(-np.pi, np.pi, 101)

plt.figure(figsize = (10,6))

plt.plot(kgrid, mono_w(kgrid))

fs = 14
plt.xlabel('$k$', fontsize=fs)
plt.ylabel('$\omega(k)$', fontsize=fs)

plt.axhline(0, color = 'k', alpha = 0.2)
plt.axhline(2, color = 'k', alpha = 0.2, linestyle = '--')

plt.axvline(-np.pi, color = 'k', alpha = 0.2)
#plt.axvline(0, color = 'k', alpha = 0.2)
plt.axvline(np.pi, color = 'k', alpha = 0.2)

plt.xticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi], ['$-\pi$', '$-\pi/2$', '$0$', '$\pi/2$', '$\pi$'])
plt.yticks([0,1, 2], ['$0$','$\sqrt{\kappa/m}$', '$2\sqrt{\kappa/m}$'])

plt.tight_layout()

plt.show()

In [ ]:
fig_out1 = widgets.Output()

N = 12  # Number of atoms
nvals = np.arange(N)
ndense = np.linspace(nvals[0], nvals[-1], 1001)
kvals = 2*np.pi/N*np.array([i for i in range(-N//2, N//2+1)])  # Include redundant mode at right BZ boundary
kdense = np.linspace(-np.pi, np.pi, 1001)

# Create widgets
kvec = widgets.FloatSlider(
    value=kvals[N//2], min=-np.pi, max=np.pi, step=2*np.pi/len(nvals),
    description="$k/a$"
)

time = widgets.FloatSlider(
    value=0, min=0, max=50, step=0.1,
    description="$t$"
)

def render1(change=None):
    t = time.value
    k = kvec.value

    y_discrete = displacement(nvals, k, t)
    y_dense = displacement(ndense, k, t)

    with fig_out1:
        fig_out1.clear_output(wait=True)

        fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(10, 4), width_ratios=[5, 3])

        ax1.plot(kvals, mono_w(kvals), 'k.')
        ax1.plot(kdense, mono_w(kdense), 'k')
        ax1.plot([k, k], [0, 2], 'r')

        ax0.plot(ndense, y_dense, lw=2)
        ax0.plot(nvals, y_discrete, 'ko')

        fs = 14
        ax0.set_xticks(nvals)
        ax0.set_xlabel('$x^{eq}/a$', fontsize=fs)
        ax0.set_ylabel('$\delta x$', fontsize=fs)
        ax0.set_ylim(-1.1, 1.1)

        plt.tight_layout()
        plt.show()

kvec.observe(render1, names="value")
time.observe(render1, names="value")

display(kvec, time, fig_out1)
render1()


In [ ]:
fig_out2 = widgets.Output()

N = 10  # Number of atoms
nvals = np.arange(N)
ndense = np.linspace(nvals[0], nvals[-1], 1001)

# Create widgets
kvec2 = widgets.FloatSlider(
    value=1, min=-np.pi, max=np.pi, step=2*np.pi/len(nvals),
    description="$k/a$"
)

time2 = widgets.FloatSlider(
    value=0, min=0, max=50, step=0.1,
    description="$t$"
)

def render2(change=None):
    t = time2.value
    k = kvec2.value

    y_discrete = displacement(nvals, k, t)
    y1 = displacement(ndense, k, t)
    y2 = displacement(ndense, k - 2*np.pi, t)

    with fig_out2:
        fig_out2.clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(8, 4))

        ax.plot(ndense, y1, lw=2, color='b')
        ax.plot(ndense, y2, lw=2, color='r')
        ax.plot(nvals, y_discrete, 'ko')

        plt.tight_layout()
        plt.show()

kvec2.observe(render2, names="value")
time2.observe(render2, names="value")

display(kvec2, time2, fig_out2)
render2()
